# Executive Data Analytics Dashboard: Retail Performance System
### Portfolio Project for Advanced AI Data Validation & Code Optimization

--- 
## 1. Project Objective & Pipeline Design
This data engineering and analytics framework programmatically simulates raw business transaction registers, executes clean ETL workflows, checks data integrity rules, and models key descriptive metrics.

* **Step 1:** Programmatic synthetic dataset generation
* **Step 2:** Data cleaning, type integrity parsing, and NaN isolation
* **Step 3:** Multi-dimensional exploratory aggregations using `Pandas`
* **Step 4:** Matrix trend plotting using `Matplotlib` and `Seaborn`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set seed for absolute testing reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")
print("Libraries successfully initialized.")

## 2. Programmatic Dataset Generation
We generate 1,500 continuous record sequences containing operational real-world edge cases (missing attributes, skewed values, categories) to prove algorithmic robust verification.

In [ ]:
records_count = 1500
start_date = datetime(2026, 1, 1)

data = {
    'Transaction_ID': [f"TXN-{1000 + i}" for i in range(records_count)],
    'Timestamp': [(start_date + timedelta(minutes=int(np.random.randint(0, 525600)))).strftime('%Y-%m-%d %H:%M:%S') for i in range(records_count)],
    'Region': np.random.choice(['Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Eldoret'], size=records_count, p=[0.4, 0.2, 0.15, 0.15, 0.10]),
    'Product_Category': np.random.choice(['Electronics', 'Office Supplies', 'Apparel', 'Hardware', 'Furniture'], size=records_count),
    'Units_Sold': np.random.randint(1, 15, size=records_count),
    'Unit_Price': np.round(np.random.uniform(15.0, 1200.0, size=records_count), 2),
    'Customer_Rating': np.round(np.random.uniform(1.0, 5.0, size=records_count), 1)
}

df_raw = pd.DataFrame(data)

# Inject synthetic Null data slots to test operational validation pipelines
df_raw.loc[np.random.choice(df_raw.index, size=45, replace=False), 'Customer_Rating'] = np.nan
df_raw.loc[np.random.choice(df_raw.index, size=15, replace=False), 'Units_Sold'] = np.nan

print(f"Raw dataset created structure: {df_raw.shape}")
df_raw.head()

## 3. Data Cleansing, Imputation & Integrity Filtering
Isolating null parameters, checking for structural integrity violations, and deriving structural features like `Gross_Revenue` calculations safely.

In [ ]:
# Copy active dataframe slice
df_clean = df_raw.copy()

# Step A: Impute volumetric transaction items with median strategies
df_clean['Units_Sold'] = df_clean['Units_Sold'].fillna(df_clean['Units_Sold'].median()).astype(int)

# Step B: Keep rating NaNs separate to prevent average score metric skews
df_clean['Customer_Rating'] = df_clean['Customer_Rating'].fillna(df_clean['Customer_Rating'].mean())

# Step C: Execute type parsing transformation schemas
df_clean['Timestamp'] = pd.to_datetime(df_clean['Timestamp'])
df_clean['Month_Period'] = df_clean['Timestamp'].dt.to_period('M')

# Step D: Formulate vector column arithmetic mapping matrices
df_clean['Gross_Revenue'] = df_clean['Units_Sold'] * df_clean['Unit_Price']

print(f"Cleaned dataset integrity verification matrix leaks remaining: {df_clean.isna().sum().sum()}")
df_clean.info()

## 4. Analytical Data Aggregations
Extracting higher-level business metrics grouped by regions and individual products.

In [ ]:
regional_performance = df_clean.groupby('Region').agg(
    Total_Orders=('Transaction_ID', 'count'),
    Total_Units=('Units_Sold', 'sum'),
    Cumulative_Revenue=('Gross_Revenue', 'sum'),
    Average_Customer_Satisfaction=('Customer_Rating', 'mean')
).sort_values(by='Cumulative_Revenue', ascending=False)

regional_performance

## 5. Visual Dashboard Matrix Generation
Plotting distribution counts and regional revenue performance curves cleanly using data visualization modules.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Regional Gross Performance Share breakdown 
sns.barplot(
    x=regional_performance.index,
    y=regional_performance['Cumulative_Revenue'],
    ax=axes[0],
    palette='Blues_r'
)
axes[0].set_title('Cumulative Revenue Contributions by Operating Hub Region', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Total Revenue Monitored')
axes[0].set_xlabel('Hub Region Location')

# Plot B: Product Category Volume Analysis
product_volume = df_clean.groupby('Product_Category')['Units_Sold'].sum().reset_index()
sns.barplot(
    x='Units_Sold',
    y='Product_Category',
    data=product_volume,
    ax=axes[1],
    palette='viridis'
)
axes[1].set_title('Consolidated Sales Units Volume Density by Product Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Aggregated Units Dispatched')
axes[1].set_ylabel('Product Line Category')

plt.tight_layout()
plt.show()